# TIME TRAVEL
- se puede consultar una tabla en su estado actual
- en una version especifica pasada
- en estado segun la fecha y hora especifica
- snapshots consistentes (cada operacion deja una foto consistente)

In [0]:
%sql DESCRIBE HISTORY users

# Consulta por fecha

In [0]:
%sql
select * from users timestamp as of '2026-11-05T22:40:55.000+00:00'

In [0]:
%sql
select * from workspace.default.users timestamp as of '2026-11-05T22:40:55.000+00:00'

#Consulta por versiones

In [0]:
%sql
select * from workspace.default.users version as of 3

#Rollback de una base de datos eliminada o recuperacion de registros para auditoria

## 1. Receuperacion por fecha

In [0]:
%sql
delete from workspace.default.users


In [0]:
%sql
describe history users

In [0]:
%sql
restore table workspace.default.users timestamp as of '2026-11-05T22:49:49.000+00:00'

In [0]:
%sql
select * from users

## 2. Recuperacion por version

In [0]:
%sql
delete from workspace.default.users

In [0]:
%sql
describe history users

In [0]:
%sql
RESTORE TABLE workspace.default.users TO VERSION AS OF 5

In [0]:
%sql
select * from users

# Optimizacion de tablas tipo delta lake
proceso de reorganizar fisicamente archivos parquet.

**Objetivo:** mejorar el rendimiento de lectura y reducir fragmentacion

### El problema
- Operaciones INSERT, UPDATE, DELETE, MERGE generan muchos archivos pequeños
- Incrementa overhead de planificacion en spark
- Degrada desempeño en consultas analiticas

#Compactacion de archivos

- Delta lake genera muchos archivos pequeños (streaming, cargas frecuentes).
- Leer 1000 archivos de 1MB es mas lento que cargar un archivo de 1GB.

### Beneficio
- Reduce overhead de metadatos y mejora la velocidad de lectura.

##Comando OPTIMIZE
- Une archivos pequeños en archivos grandes eficientes (~1GB).

###Proceso interno

**1.** lee muchos archivos pequeños.

**2.** los combina.

**3.** escribe nuevos archivos mas grandes (~128MB).

**4.** Marca los pequeños como removed en el log.

**Es compactacion inteligente. No cambia los datos, solo los reorganiza.**

In [0]:
%sql
OPTIMIZE workspace.default.users

# ¿Porque la fragmentacion es mala?
**Spark procesa particiones y tareas, no archivos**

### Si tienes muchos parquets pequeños puede resultar en: 
- Mas metadata.
- Mas overhead.
- Peor performance.
- Mas tasks.
- Mas tiempo de planificacion.

**el problema no es el tamaño total, es la fragmentacion**

# ¿Que es el Z-ORDER?

**Tecnica para reorganizar fisicamente datos dentro de archivos parquet**

- Valores similares quedan cerca entre si.

**NO ES**

- un indice
- una particion
- una tabla nueva

_es reordenamiento fisico del almacenamiento_

## Ejemplo de Z-ORDER

![image_1793944091212.png](./image_1793944091212.png "image_1793944091212.png")

Al aplicar la siguiente sentencia:

`OPTIMIZE tabla ZORDER BY (customer_id);`

da como resultado la siguiente separacion de registros:

antes:

- Parquet A -> 50, 2000, 51, 1999 `

despues:

- Parquet A -> 50, 51 | Parquet B -> 1999, 2000

esto con el fin de que al ejecutar el siguiente query:

`select * from tabla where customer_id = 50;`

el tiempo de ejecucion de la query se reduce debido a que solo va a realizar la busqueda en el parquet A.

_Query mas rapida al leer menos datos_
_Z-ORDER co-localiza datos relacionados para mejorar la eficiencia de la omision de datos_

# Limites del Z-ORDER

**limite practico**

- En produccion: 1 a 3 columnas maximo
- 4 columnas ya disminuye el beneficio

**Muchas columnas**

- Pierde efectividad
- Disperza la localizacion
- No mejora el data skipping

**usarlo en:**

- Columnas de alta cardinalidad
- Columnas frecuentes en filtros
- Columnas usadas juntas en queries

**Ejemplo bueno**

`ZORDER BY (customer_id, fecha)`

_Alta cardinalidad, frecuentes en filtros_

**ejemplo malo**

`ZORDER BY (pais, genero)`

_Baja cardinalidad, poco beneficio (si tienen pocos resultados)_